# Full Model Evaluation v2 — Flattened `questions`/`answers` schema

Fixes v1 schema bug: `generated_v4style_300.json` contains arrays `questions` and `answers`, so each raw row must be flattened into separate evaluable samples. This notebook prints full metrics and saves predictions, summary, per-label metrics, confusion matrix, and errors.

In [ ]:
# ============================================================
# Config
# ============================================================
import os, json, re, time, math, traceback, gc, csv
from pathlib import Path
from collections import Counter, defaultdict

DATA_PATH = "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json"
TEST_PATH = "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json"
OUT_DIR = "/kaggle/working"

BASE_MODEL_PATH = None   # auto-detect if None
ADAPTER_PATH = None      # auto-detect if None

MAX_NEW_TOKENS = 192
DO_SAMPLE = False
TEMPERATURE = None
TOP_P = None
LIMIT = None             # set e.g. 20 for smoke test
SAVE_EVERY = 25
RESUME = True

PRED_PATH = os.path.join(OUT_DIR, "full_model_eval_v2_flatten_preds.json")
SUMMARY_PATH = os.path.join(OUT_DIR, "full_model_eval_v2_flatten_summary.json")
ERROR_PATH = os.path.join(OUT_DIR, "full_model_eval_v2_flatten_error_cases.json")
PER_LABEL_CSV = os.path.join(OUT_DIR, "full_model_eval_v2_flatten_per_label_metrics.csv")
CONFUSION_CSV = os.path.join(OUT_DIR, "full_model_eval_v2_flatten_confusion_matrix.csv")

print('DATA_PATH:', DATA_PATH, 'exists=', os.path.exists(DATA_PATH))
print('TEST_PATH:', TEST_PATH, 'exists=', os.path.exists(TEST_PATH))
print('OUT_DIR:', OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# Helpers: loading, flattening, label parsing
# ============================================================
LABELS_ALL = ['A','B','C','D','Yes','No','Unknown']
MC_LABELS = ['A','B','C','D']
YNU_LABELS = ['Yes','No','Unknown']
FINAL_RE = re.compile(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', re.I)
ANY_LABEL_RE = re.compile(r'\b(Yes|No|Unknown|[ABCD])\b', re.I)

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def normalize_label(x):
    if x is None:
        return None
    s = str(x).strip()
    if not s:
        return None
    u = s.upper()
    if u in MC_LABELS:
        return u
    t = s.title()
    if t in YNU_LABELS:
        return t
    return None

def parse_answer(text):
    if text is None:
        return 'UNPARSEABLE'
    s = str(text)
    matches = FINAL_RE.findall(s)
    if matches:
        return normalize_label(matches[-1]) or 'UNPARSEABLE'
    # fallback: final label near end
    tail = s[-500:]
    matches = ANY_LABEL_RE.findall(tail)
    if matches:
        return normalize_label(matches[-1]) or 'UNPARSEABLE'
    return 'UNPARSEABLE'

def flatten_rows(raw_rows, split_name='test'):
    flat, bad = [], []
    for row_i, row in enumerate(raw_rows):
        qs = row.get('questions', [])
        ans = row.get('answers', [])
        exps = row.get('explanation', [])
        if isinstance(qs, str): qs = [qs]
        if isinstance(ans, str): ans = [ans]
        if isinstance(exps, str): exps = [exps]
        if not isinstance(qs, list) or not isinstance(ans, list):
            bad.append((row_i, 'questions/answers not list or str'))
            continue
        n = max(len(qs), len(ans))
        for j in range(n):
            q = qs[j] if j < len(qs) else ''
            gold = normalize_label(ans[j] if j < len(ans) else None)
            idx = row.get('idx')
            sample = {
                'flat_id': f'{split_name}_{row_i}_{j}',
                'row_i': row_i,
                'q_i': j,
                'idx': idx,
                'idx_component': idx[j] if isinstance(idx, list) and j < len(idx) else None,
                'premises_NL': row.get('premises-NL') or row.get('premises_NL') or [],
                'premises_FOL': row.get('premises-FOL') or row.get('premises_FOL') or [],
                'question': str(q or '').strip(),
                'gold': gold,
                'gold_raw': ans[j] if j < len(ans) else None,
                'gold_explanation': exps[j] if isinstance(exps, list) and j < len(exps) else None,
                'is_mc': gold in MC_LABELS,
                'is_ynu': gold in YNU_LABELS,
                'source': split_name,
            }
            flat.append(sample)
    return flat, bad

raw_data = load_json(DATA_PATH)
raw_test = load_json(TEST_PATH)
print('Loaded raw data rows:', len(raw_data), 'raw test rows:', len(raw_test))
print('First test keys:', list(raw_test[0].keys()))
flat_test, bad = flatten_rows(raw_test, 'test')
if LIMIT:
    flat_test = flat_test[:LIMIT]
print('Flattened test samples:', len(flat_test), 'bad rows:', len(bad))
print('First flat sample:')
print(json.dumps({k: flat_test[0][k] for k in ['flat_id','question','gold','is_mc','is_ynu','idx_component']}, ensure_ascii=False, indent=2))
print('Gold distribution:', Counter(s['gold'] or 'NO_GOLD' for s in flat_test))
print('MC:', sum(s['is_mc'] for s in flat_test), 'YNU:', sum(s['is_ynu'] for s in flat_test))
assert all(s['question'] for s in flat_test), 'Some flattened questions are empty — schema parsing still wrong.'
assert sum(s['gold'] is not None for s in flat_test) > 0, 'No gold labels found after flattening.'

In [ ]:
# ============================================================
# Prompt builder
# ============================================================
def format_premises(premises, max_items=None):
    if not isinstance(premises, list):
        premises = [str(premises)]
    if max_items:
        premises = premises[:max_items]
    return '\n'.join(f'{i+1}. {p}' for i, p in enumerate(premises))

def build_prompt(sample):
    premises = sample.get('premises_NL') or []
    q = sample['question']
    if sample['is_mc']:
        instruction = 'End with exactly one line: Final Answer: <A, B, C, or D>'
    else:
        instruction = 'End with exactly one line: Final Answer: <Yes, No, or Unknown>'
    return f"""You are solving a logic-based educational QA problem. Use only the given premises. Do not use outside knowledge.

Premises:
{format_premises(premises)}

Question:
{q}

Reason step by step briefly, cite supporting premises if useful, and {instruction}
"""

print('Prompt preview:')
print(build_prompt(flat_test[0])[:1800])

In [ ]:
# ============================================================
# Model loading
# ============================================================
def find_candidates():
    model_cands, adapter_cands = [], []
    for root in ['/kaggle/input']:
        if not os.path.exists(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            files = set(filenames)
            if 'config.json' in files and ('model.safetensors.index.json' in files or any(f.endswith('.safetensors') for f in files)):
                model_cands.append(dirpath)
            if 'adapter_config.json' in files:
                adapter_cands.append(dirpath)
    return model_cands, adapter_cands

model_cands, adapter_cands = find_candidates()
print('Model candidates top 10:')
for x in model_cands[:10]: print(' ', x)
print('Adapter candidates top 10:')
for x in adapter_cands[:10]: print(' ', x)

if BASE_MODEL_PATH is None:
    qwen = [p for p in model_cands if 'qwen' in p.lower() and ('8b' in p.lower() or '8-b' in p.lower())]
    BASE_MODEL_PATH = qwen[0] if qwen else (model_cands[0] if model_cands else None)
if ADAPTER_PATH is None:
    pref = [p for p in adapter_cands if 'v2333333' in p.lower() or 'yixuan' in p.lower()]
    ADAPTER_PATH = pref[0] if pref else (adapter_cands[0] if adapter_cands else None)
print('Using BASE_MODEL_PATH:', BASE_MODEL_PATH)
print('Using ADAPTER_PATH:', ADAPTER_PATH)
assert BASE_MODEL_PATH and os.path.exists(BASE_MODEL_PATH), 'Base model path not found. Set BASE_MODEL_PATH manually.'

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
load_error, load_mode = None, None
try:
    tok_path = ADAPTER_PATH if ADAPTER_PATH and os.path.exists(os.path.join(ADAPTER_PATH, 'tokenizer_config.json')) else BASE_MODEL_PATH
    tok = AutoTokenizer.from_pretrained(tok_path, trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_PATH,
        dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    if ADAPTER_PATH and os.path.exists(os.path.join(ADAPTER_PATH, 'adapter_config.json')):
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, ADAPTER_PATH)
        load_mode = 'base_plus_adapter_sharded_dtype'
    else:
        load_mode = 'base_only_sharded_dtype'
    model.eval()
except Exception:
    load_error = traceback.format_exc()
    print(load_error)
    raise
print('MODEL LOADED:', load_mode)

In [ ]:
# ============================================================
# Generation loop
# ============================================================
def model_first_device(model):
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

def generate_one(prompt):
    inputs = tok(prompt, return_tensors='pt')
    dev = model_first_device(model)
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    gen_kwargs = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE,
                      pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id)
    if DO_SAMPLE:
        if TEMPERATURE is not None: gen_kwargs['temperature'] = TEMPERATURE
        if TOP_P is not None: gen_kwargs['top_p'] = TOP_P
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True)

existing = {}
if RESUME and os.path.exists(PRED_PATH):
    try:
        old = load_json(PRED_PATH)
        existing = {r['flat_id']: r for r in old if 'flat_id' in r}
        print('Resume: loaded existing preds:', len(existing))
    except Exception as e:
        print('Could not resume:', e)

preds = []
start_all = time.time()
for i, sample in enumerate(flat_test, 1):
    if sample['flat_id'] in existing:
        preds.append(existing[sample['flat_id']])
        continue
    prompt = build_prompt(sample)
    t0 = time.time()
    try:
        raw = generate_one(prompt)
        pred = parse_answer(raw)
        err = None
    except Exception:
        raw = ''
        pred = 'UNPARSEABLE'
        err = traceback.format_exc()
    lat = time.time() - t0
    rec = dict(sample)
    rec.update({'prompt': prompt, 'raw_output': raw, 'pred': pred, 'latency_sec': lat, 'error': err})
    preds.append(rec)
    if i == 1 or i % 10 == 0:
        done_new = sum(1 for r in preds if r.get('latency_sec') is not None)
        avg = sum(r.get('latency_sec') or 0 for r in preds) / max(1, done_new)
        print(f'[{i}/{len(flat_test)}] pred={pred} gold={sample["gold"]} mc={sample["is_mc"]} avg_latency={avg:.2f}s')
    if i % SAVE_EVERY == 0:
        with open(PRED_PATH, 'w', encoding='utf-8') as f:
            json.dump(preds, f, ensure_ascii=False, indent=2)
with open(PRED_PATH, 'w', encoding='utf-8') as f:
    json.dump(preds, f, ensure_ascii=False, indent=2)
print('Total inference time sec:', time.time() - start_all)
print('Saved preds:', PRED_PATH)

In [ ]:
# ============================================================
# Metrics
# ============================================================
def safe_div(a,b): return a/b if b else 0.0

def compute_metrics(rows, labels):
    rows = [r for r in rows if normalize_label(r.get('gold')) in labels]
    n = len(rows)
    correct = sum(1 for r in rows if normalize_label(r.get('gold')) == normalize_label(r.get('pred')))
    per = {}
    for lab in labels:
        tp = sum(1 for r in rows if r.get('gold') == lab and r.get('pred') == lab)
        fp = sum(1 for r in rows if r.get('gold') != lab and r.get('pred') == lab)
        fn = sum(1 for r in rows if r.get('gold') == lab and r.get('pred') != lab)
        prec = safe_div(tp, tp+fp)
        rec = safe_div(tp, tp+fn)
        f1 = safe_div(2*prec*rec, prec+rec)
        per[lab] = dict(tp=tp, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1,
                        support=sum(1 for r in rows if r.get('gold') == lab),
                        pred_count=sum(1 for r in rows if r.get('pred') == lab))
    macro = sum(per[l]['f1'] for l in labels) / len(labels) if labels else None
    total_support = sum(per[l]['support'] for l in labels)
    weighted = safe_div(sum(per[l]['f1'] * per[l]['support'] for l in labels), total_support) if labels else None
    micro_f1 = safe_div(correct, n)
    cm = {g: {p: 0 for p in labels + ['UNPARSEABLE','OTHER']} for g in labels}
    for r in rows:
        g = r.get('gold')
        p = r.get('pred')
        if p not in labels:
            p = 'UNPARSEABLE' if p == 'UNPARSEABLE' else 'OTHER'
        cm[g][p] += 1
    return dict(n=n, correct=correct, acc=safe_div(correct,n) if n else None, micro_f1=micro_f1 if n else None,
                macro_f1=macro, weighted_f1=weighted, labels=labels, per_label=per, confusion_matrix=cm)

metrics_all = compute_metrics(preds, LABELS_ALL)
metrics_mc = compute_metrics(preds, MC_LABELS)
metrics_ynu = compute_metrics(preds, YNU_LABELS)
latencies = [r.get('latency_sec') for r in preds if isinstance(r.get('latency_sec'), (int,float))]

def err_type(r): return f"{r.get('gold')}->{r.get('pred')}"
errors = [r for r in preds if r.get('gold') and r.get('pred') != r.get('gold')]
error_cases = [{k: r.get(k) for k in ['flat_id','row_i','q_i','idx','idx_component','gold','pred','is_mc','question','gold_explanation','raw_output','latency_sec']} for r in errors]

summary = {
    'version': 'full_model_eval_v2_flatten',
    'data_path': DATA_PATH,
    'test_path': TEST_PATH,
    'base_model_path': BASE_MODEL_PATH,
    'adapter_path': ADAPTER_PATH,
    'load_mode': load_mode,
    'load_error': load_error,
    'raw_test_rows': len(raw_test),
    'flattened_samples': len(flat_test),
    'gold_available': sum(1 for r in flat_test if r.get('gold')),
    'mc_count': sum(1 for r in flat_test if r.get('is_mc')),
    'ynu_count': sum(1 for r in flat_test if r.get('is_ynu')),
    'pred_distribution': dict(Counter(r.get('pred') for r in preds)),
    'gold_distribution': dict(Counter(r.get('gold') or 'NO_GOLD' for r in preds)),
    'invalid_unparseable_count': sum(1 for r in preds if r.get('pred') == 'UNPARSEABLE'),
    'invalid_unparseable_rate': safe_div(sum(1 for r in preds if r.get('pred') == 'UNPARSEABLE'), len(preds)),
    'latency_sec': {'total': sum(latencies), 'mean': safe_div(sum(latencies), len(latencies)), 'min': min(latencies) if latencies else None, 'max': max(latencies) if latencies else None},
    'metrics': {'all': metrics_all, 'mc': metrics_mc, 'ynu': metrics_ynu},
    'error_count': len(error_cases),
    'error_rate': safe_div(len(error_cases), len([r for r in preds if r.get('gold')])),
    'error_type_counts': dict(Counter(err_type(r) for r in errors)),
}
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f: json.dump(summary, f, ensure_ascii=False, indent=2)
with open(ERROR_PATH, 'w', encoding='utf-8') as f: json.dump(error_cases, f, ensure_ascii=False, indent=2)
print('Saved summary:', SUMMARY_PATH)
print('Saved error cases:', ERROR_PATH)
print('\n=== SUMMARY ===')
print(json.dumps({
    'flattened_samples': summary['flattened_samples'],
    'gold_available': summary['gold_available'],
    'mc_count': summary['mc_count'],
    'ynu_count': summary['ynu_count'],
    'pred_distribution': summary['pred_distribution'],
    'gold_distribution': summary['gold_distribution'],
    'invalid_unparseable_rate': summary['invalid_unparseable_rate'],
    'latency_sec': summary['latency_sec'],
    'all_metrics': {k: metrics_all[k] for k in ['n','correct','acc','micro_f1','macro_f1','weighted_f1']},
    'mc_metrics': {k: metrics_mc[k] for k in ['n','correct','acc','micro_f1','macro_f1','weighted_f1']},
    'ynu_metrics': {k: metrics_ynu[k] for k in ['n','correct','acc','micro_f1','macro_f1','weighted_f1']},
    'error_count': summary['error_count'],
    'error_type_counts': summary['error_type_counts'],
}, ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# Save CSV tables and display compact reports
# ============================================================
with open(PER_LABEL_CSV, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['split','label','precision','recall','f1','support','pred_count','tp','fp','fn'])
    for split, met in [('all', metrics_all), ('mc', metrics_mc), ('ynu', metrics_ynu)]:
        for lab, d in met['per_label'].items():
            w.writerow([split, lab, d['precision'], d['recall'], d['f1'], d['support'], d['pred_count'], d['tp'], d['fp'], d['fn']])

with open(CONFUSION_CSV, 'w', newline='', encoding='utf-8') as f:
    labels = LABELS_ALL + ['UNPARSEABLE','OTHER']
    w = csv.writer(f)
    w.writerow(['gold\\pred'] + labels)
    cm = metrics_all['confusion_matrix']
    for g in LABELS_ALL:
        w.writerow([g] + [cm.get(g, {}).get(p, 0) for p in labels])

print('Saved per-label CSV:', PER_LABEL_CSV)
print('Saved confusion CSV:', CONFUSION_CSV)
print('\n=== PER LABEL: ALL ===')
for lab, d in metrics_all['per_label'].items():
    print(f"{lab:8s} P={d['precision']:.4f} R={d['recall']:.4f} F1={d['f1']:.4f} support={d['support']} pred={d['pred_count']}")
print('\n=== CONFUSION MATRIX: ALL ===')
print('gold\\pred', LABELS_ALL + ['UNPARSEABLE','OTHER'])
for g, row in metrics_all['confusion_matrix'].items():
    print(g, [row.get(p,0) for p in LABELS_ALL + ['UNPARSEABLE','OTHER']])
print('\n=== TOP ERROR TYPES ===')
for k,v in Counter(err_type(r) for r in errors).most_common(20):
    print(k, v)
print('\n=== ERROR PREVIEW ===')
for e in error_cases[:10]:
    print('---')
    print('flat_id:', e['flat_id'], 'gold:', e['gold'], 'pred:', e['pred'], 'is_mc:', e['is_mc'])
    print('Q:', e['question'][:300])
    print('RAW:', (e.get('raw_output') or '')[:500])